# Lab 2 — Orchestration for a Loan Processing Pipeline (Azure OpenAI)

In this lab, you will implement an orchestrated **loan-processing pipeline** using **Azure OpenAI** where multiple AI-driven tasks run in a controlled, end-to-end workflow. Instead of a single prompt-response interaction, you will design a pipeline that performs stages such as intake, document/field checks, eligibility assessment, risk summarization, and decision packaging—while enforcing **clear step boundaries, branching logic, retries, and an audit trail**. By the end, you will have a reusable orchestration pattern for building production-style AI workflows that are predictable, traceable, and easy to extend with additional checks or downstream integrations.


## 0) Setup
Install packages if needed and initialize the AzureOpenAI client (same as Lab 1).

In [1]:
!pip -q install openai numpy


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
from openai import AzureOpenAI

AZURE_OPENAI_ENDPOINT = os.getenv(
    "AZURE_OPENAI_ENDPOINT", "https://agenticaiengineer.openai.azure.com/"
)
AZURE_OPENAI_API_KEY = os.environ.get(
    "AZURE_OPENAI_API_KEY",
    "<REPLACE_WITH_YOUR_AZURE_OPENAI_KEY>",
)
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

if not AZURE_OPENAI_API_KEY:
    raise ValueError(
        "Missing AZURE_OPENAI_API_KEY env var. Set it securely (do not hardcode keys)."
    )

CHAT_DEPLOYMENT = "gpt-4o-mini"

client = AzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)

print("AzureOpenAI client ready")

AzureOpenAI client ready


## 1) Core helper: a safe chat wrapper (with basic retry)
Orchestrations fail in real life: network blips, timeouts, rate limits. We’ll add a simple retry loop.

In [5]:
import time
from typing import List, Dict, Any, Optional
import json


def chat(
    messages: List[Dict[str, str]],
    *,
    temperature: float = 0.2,
    max_tokens: int = 600,
    retries: int = 3,
) -> str:
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model=CHAT_DEPLOYMENT,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            sleep_s = 1.5**attempt
            print(
                f"⚠️ chat attempt {attempt} failed: {type(e).__name__}. Retrying in {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)
    raise last_err

## 2) Define pipeline state + audit log
We keep all inputs, outputs, and decisions in a single state object.

In [6]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Literal

Decision = Literal["APPROVE", "MANUAL_REVIEW", "REJECT"]


@dataclass
class LoanState:
    application_id: str
    applicant_text: str
    applicant_record: dict = field(default_factory=dict)
    extracted_docs: dict = field(default_factory=dict)
    credit_result: dict = field(default_factory=dict)
    fraud_result: dict = field(default_factory=dict)
    risk_result: dict = field(default_factory=dict)
    decision: Optional[Decision] = None
    decision_reason: Optional[str] = None
    letter: Optional[str] = None
    audit: list = field(default_factory=list)

    def log(self, step: str, payload: dict):
        self.audit.append(
            {
                "ts": datetime.utcnow().isoformat() + "Z",
                "step": step,
                "payload": payload,
            }
        )


state = LoanState(
    application_id="APP-1001",
    applicant_text="Name: Riya Sharma. Age 29. Employed full-time. Income 120000 THB/month. Debt 25000 THB/month. Loan 300000 THB for home renovation, term 24 months.",
)
state

LoanState(application_id='APP-1001', applicant_text='Name: Riya Sharma. Age 29. Employed full-time. Income 120000 THB/month. Debt 25000 THB/month. Loan 300000 THB for home renovation, term 24 months.', applicant_record={}, extracted_docs={}, credit_result={}, fraud_result={}, risk_result={}, decision=None, decision_reason=None, letter=None, audit=[])

## 3) Step: Normalize intake into JSON
We make the model convert free text to a schema. This makes downstream steps deterministic.

In [7]:
SYSTEM_NORMALIZE = """Extract a loan applicant record from text.
Return ONLY valid JSON with keys:
full_name, age, employment, monthly_income, monthly_debt, loan_amount, term_months, purpose
Use numbers for numeric fields. If unknown, set null.
"""


def step_normalize_intake(state: LoanState) -> LoanState:
    msgs = [
        {"role": "system", "content": SYSTEM_NORMALIZE},
        {"role": "user", "content": state.applicant_text},
    ]
    out = chat(msgs, temperature=0)
    record = json.loads(out)
    state.applicant_record = record
    state.log("normalize_intake", {"record": record})
    return state


state = step_normalize_intake(state)
state.applicant_record

{'full_name': 'Riya Sharma',
 'age': 29,
 'employment': 'full-time',
 'monthly_income': 120000,
 'monthly_debt': 25000,
 'loan_amount': 300000,
 'term_months': 24,
 'purpose': 'home renovation'}

## 4) Step: Deterministic scoring (DTI + basic rules)
Keep critical math and thresholds in code so behavior is stable and auditable.

In [8]:
def compute_dti(rec: dict) -> Optional[float]:
    inc = rec.get("monthly_income")
    debt = rec.get("monthly_debt")
    if inc in (None, 0) or debt is None:
        return None
    return float(debt) / float(inc)


def step_compute_risk(state: LoanState) -> LoanState:
    rec = state.applicant_record
    dti = compute_dti(rec)

    if rec.get("age") is not None and rec["age"] < 18:
        decision = "REJECT"
        band = "ineligible"
        reason = "Applicant is under 18."
    elif dti is None:
        decision = "MANUAL_REVIEW"
        band = "unknown"
        reason = "Missing financial inputs to compute DTI."
    elif dti <= 0.35:
        decision = "APPROVE"
        band = "low"
        reason = f"DTI {dti:.2f} is within low-risk range."
    elif dti <= 0.50:
        decision = "MANUAL_REVIEW"
        band = "medium"
        reason = f"DTI {dti:.2f} indicates medium risk."
    else:
        decision = "MANUAL_REVIEW"
        band = "high"
        reason = f"DTI {dti:.2f} indicates high risk."

    state.risk_result = {"dti": dti, "band": band}
    state.log("compute_risk", state.risk_result)
    # decision is provisional; later checks (fraud/credit) can override
    state.decision = decision
    state.decision_reason = reason
    state.log("provisional_decision", {"decision": decision, "reason": reason})
    return state


state = step_compute_risk(state)
state.risk_result, state.decision, state.decision_reason

({'dti': 0.20833333333333334, 'band': 'low'},
 'APPROVE',
 'DTI 0.21 is within low-risk range.')

## 5) Steps: Credit check + Fraud check (mocked)
In real solutions, these call internal services. Here we simulate them to focus on orchestration.

In [9]:
import random


def step_credit_check(state: LoanState) -> LoanState:
    # mock score: 300-850
    score = random.randint(580, 790)
    bucket = "good" if score >= 700 else "fair" if score >= 650 else "poor"
    state.credit_result = {"score": score, "bucket": bucket}
    state.log("credit_check", state.credit_result)
    return state


def step_fraud_check(state: LoanState) -> LoanState:
    # mock signals
    flags = []
    if (
        state.applicant_record.get("loan_amount", 0)
        and state.applicant_record["loan_amount"] > 500000
    ):
        flags.append("high_amount")
    risk = "clear" if not flags else "flagged"
    state.fraud_result = {"risk": risk, "flags": flags}
    state.log("fraud_check", state.fraud_result)
    return state


state = step_credit_check(state)
state = step_fraud_check(state)
state.credit_result, state.fraud_result

({'score': 583, 'bucket': 'poor'}, {'risk': 'clear', 'flags': []})

## 6) Orchestrate checks in parallel (optional)
If your environment supports it, run independent steps concurrently.

In [10]:
import asyncio


async def run_parallel_checks(state: LoanState) -> LoanState:
    loop = asyncio.get_event_loop()
    # Wrap sync functions
    await loop.run_in_executor(None, step_credit_check, state)
    await loop.run_in_executor(None, step_fraud_check, state)
    return state


# Uncomment to try parallel execution:
# state = asyncio.run(run_parallel_checks(state))
# state.credit_result, state.fraud_result

## 7) Final decision logic (combine risk + checks)
We’ll add simple deterministic overrides:
- Fraud flagged → MANUAL_REVIEW
- Credit poor + medium/high risk → REJECT (example rule)

Then we’ll ask the model to generate a **clear explanation** for the outcome.

In [11]:
SYSTEM_EXPLAIN = """You are a loan operations assistant.
Write a short, professional explanation for the final decision.
Use only the provided facts. Do not invent.
"""


def finalize_decision(state: LoanState) -> LoanState:
    band = state.risk_result.get("band")
    credit_bucket = state.credit_result.get("bucket")
    fraud_risk = state.fraud_result.get("risk")

    decision = state.decision or "MANUAL_REVIEW"
    reasons = [state.decision_reason] if state.decision_reason else []

    if fraud_risk == "flagged":
        decision = "MANUAL_REVIEW"
        reasons.append("Fraud check raised flags that require a manual review.")
    elif credit_bucket == "poor" and band in ("medium", "high"):
        decision = "REJECT"
        reasons.append(
            "Credit score and risk indicators do not meet automated approval thresholds."
        )

    state.decision = decision
    state.decision_reason = " ".join([r for r in reasons if r])

    state.log(
        "final_decision", {"decision": state.decision, "reason": state.decision_reason}
    )
    return state


def step_generate_explanation(state: LoanState) -> LoanState:
    facts = {
        "applicant_record": state.applicant_record,
        "risk_result": state.risk_result,
        "credit_result": state.credit_result,
        "fraud_result": state.fraud_result,
        "decision": state.decision,
        "decision_reason": state.decision_reason,
    }
    msgs = [
        {"role": "system", "content": SYSTEM_EXPLAIN},
        {"role": "user", "content": json.dumps(facts, indent=2)},
    ]
    explanation = chat(msgs, temperature=0.2, max_tokens=220)
    state.log("llm_explanation", {"text": explanation})
    return state


state = finalize_decision(state)
state = step_generate_explanation(state)
state.decision, state.decision_reason

('APPROVE', 'DTI 0.21 is within low-risk range.')

## 8) Generate a decision letter (customer-facing)
We generate a polite letter. In production, include compliance-approved templates.

In [12]:
SYSTEM_LETTER = """You write a short loan decision letter.
Requirements:
- professional tone
- 8-12 lines
- include application_id
- do not include sensitive internal thresholds
- do not invent missing details
"""


def step_generate_letter(state: LoanState) -> LoanState:
    payload = {
        "application_id": state.application_id,
        "decision": state.decision,
        "reason": state.decision_reason,
        "purpose": state.applicant_record.get("purpose"),
        "loan_amount": state.applicant_record.get("loan_amount"),
        "term_months": state.applicant_record.get("term_months"),
    }
    msgs = [
        {"role": "system", "content": SYSTEM_LETTER},
        {"role": "user", "content": json.dumps(payload, indent=2)},
    ]
    letter = chat(msgs, temperature=0.2, max_tokens=260)
    state.letter = letter
    state.log("decision_letter", {"text": letter})
    return state


state = step_generate_letter(state)
print(state.letter)

[Your Company Letterhead]  
[Date]  

[Applicant's Name]  
[Applicant's Address]  
[City, State, Zip Code]  

Dear [Applicant's Name],  

We are pleased to inform you that your loan application (Application ID: APP-1001) for a home renovation has been approved. After careful review, we have determined that your debt-to-income ratio of 0.21 falls within a low-risk range, which supports our decision to grant your request.  

You have been approved for a loan amount of $300,000 with a term of 24 months. Please review the attached documents for further details regarding the terms and conditions of your loan.  

Should you have any questions or require additional information, please do not hesitate to contact us.  

Thank you for choosing [Your Company Name].  

Sincerely,  
[Your Name]  
[Your Title]  
[Your Company Name]  
[Your Contact Information]  


## 9) Audit output
A key orchestration requirement is to keep an audit trail. Here we print a compact audit log.

In [13]:
for item in state.audit:
    print(item["ts"], "-", item["step"])

2025-12-15T16:22:40.486557Z - normalize_intake
2025-12-15T16:22:40.497198Z - compute_risk
2025-12-15T16:22:40.497202Z - provisional_decision
2025-12-15T16:22:40.504694Z - credit_check
2025-12-15T16:22:40.504712Z - fraud_check
2025-12-15T16:22:45.493534Z - final_decision
2025-12-15T16:22:49.672547Z - llm_explanation
2025-12-15T16:22:54.987084Z - decision_letter
